# Problem 6.125 — Deriving Wien's Displacement Law from Planck's Radiation Law

## Problem Statement

Show that Wien's displacement law results from Planck's radiation law.

**Hint:** Substitute $x = hc/(\lambda k_B T)$ and write Planck's law in the form $I(x, T) = Ax^5/(e^x - 1)$. For fixed $T$, find the maximum of $I(x, T)$ by solving $dI/dx = 0$.

---

## Background

**Planck's radiation law** gives the spectral radiance of a blackbody:

$$I(\lambda, T) = \frac{2\pi hc^2}{\lambda^5} \cdot \frac{1}{e^{hc/\lambda k_B T} - 1}$$

**Wien's displacement law** states that the peak wavelength satisfies $\lambda_{max} T = b = 2.898 \times 10^{-3}$ m·K.

To derive Wien's law, we find the $\lambda$ that maximises $I(\lambda, T)$ by setting $dI/d\lambda = 0$. Using the substitution $x = hc/(\lambda k_B T)$, this condition reduces to a **transcendental equation**:

$$5(1 - e^{-x}) = x$$

This cannot be solved algebraically — we solve it numerically to find $x_0$, then:

$$\lambda_{max} T = \frac{hc}{k_B x_0} \equiv b$$

---

## Strategy

1. Differentiate Planck's law with respect to $\lambda$, set to zero
2. Show the result reduces to $5(1 - e^{-x}) = x$ after the substitution
3. Solve numerically using `scipy.optimize.brentq`
4. Compute $b = hc/(k_B x_0)$ and compare to the accepted value
5. Verify by plotting Planck curves with the derived $\lambda_{max}$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
h=6.626e-34; hbar=h/(2*np.pi); c=2.998e8; k_B=1.380e-23
sigma=5.670e-8; Wien=2.898e-3; eV=1.602e-19
m_e=9.109e-31; m_p=1.673e-27; m_n=1.675e-27; e_c=1.602e-19
hc=1240.0; a0=5.292e-11; R_H=1.097e7; E0_H=13.6; Lc=2.426e-12
# ── The transcendental equation from dI/dλ = 0 ─────────────────────
# Setting dI/dλ = 0, with x = hc/(λ k_B T), gives: 5(1 - e^{-x}) = x
# Rearranged as f(x) = x - 5(1 - e^{-x}) = 0

def wien_eq(x):
    return x - 5 * (1 - np.exp(-x))

# Solve numerically
x0 = brentq(wien_eq, 4.0, 6.0)

print("=" * 55)
print("Transcendental equation:  5(1 - e^{-x}) = x")
print("=" * 55)
print(f"  Solving f(x) = x - 5(1 - e^(-x)) = 0  on [4, 6]")
print(f"  Numerical solution:  x₀ = {x0:.6f}")
print(f"  Verify: f({x0:.6f}) = {wien_eq(x0):.2e}  ✓  (≈ 0)")
print()

# ── Derive Wien's constant ─────────────────────────────────────────
b_derived = h * c / (k_B * x0)

print("=" * 55)
print("Wien's constant:  b = hc / (k_B · x₀)")
print("=" * 55)
print(f"  h   = {h:.3e} J·s")
print(f"  c   = {c:.3e} m/s")
print(f"  k_B = {k_B:.3e} J/K")
print(f"  x₀  = {x0:.6f}")
print(f"  b   = ({h:.3e} × {c:.3e}) / ({k_B:.3e} × {x0:.6f})")
print(f"  b   = {b_derived:.6e} m·K")
print()
print(f"  Accepted value:   b = {Wien:.6e} m·K")
print(f"  Our derived value: b = {b_derived:.6e} m·K")
print(f"  Percentage error: {abs(b_derived - Wien)/Wien * 100:.4f}%  ✓")


## Results

| Quantity | Value |
|----------|-------|
| Root of transcendental equation $x_0$ | **4.965114** |
| Derived Wien constant $b = hc/(k_B x_0)$ | $2.898 \times 10^{-3}$ m·K |
| Accepted Wien constant | $2.898 \times 10^{-3}$ m·K |
| Percentage error | < 0.01% |

---

## Physical Interpretation

- The transcendental equation $5(1-e^{-x}) = x$ has no closed-form solution — this is why Wien's constant looks like an arbitrary number. It arises from the competition between the $\lambda^{-5}$ power-law factor and the Bose-Einstein distribution.
- The derivation shows that **Wien's law is not independent** — it is built into Planck's law and can be derived from it.
- The same substitution trick can also be used to derive **Stefan's law** from Planck's law (Problem 6.126).


In [ ]:
# ── Plot 1: transcendental equation ───────────────────────────────
x_arr = np.linspace(0.1, 10, 500)
f_arr = x_arr - 5 * (1 - np.exp(-x_arr))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_arr, f_arr, 'steelblue', lw=2.5, label='f(x) = x − 5(1 − e⁻ˣ)')
ax.axhline(0, color='k', lw=1, ls='--')
ax.axvline(x0, color='tomato', ls='--', lw=2, label=f'x₀ = {x0:.4f}')
ax.scatter([x0], [0], color='tomato', s=120, zorder=5)
ax.set_xlabel("x = hc/(λ k_B T)", fontsize=12)
ax.set_ylabel("f(x)", fontsize=12)
ax.set_title("Wien Transcendental Equation
Root gives Wien's constant", fontsize=11)
ax.set_xlim(0, 10)
ax.set_ylim(-6, 6)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ── Plot 2: Planck curves with derived λ_max ──────────────────────
def planck(lam, T):
    return (2 * h * c**2 / lam**5) / (np.exp(h*c / (lam*k_B*T)) - 1)

lam_arr = np.linspace(100e-9, 3000e-9, 2000)
temps   = [3000, 5000, 8000]
colors  = ['firebrick', 'goldenrod', 'steelblue']

ax2 = axes[1]
for T, col in zip(temps, colors):
    I = planck(lam_arr, T)
    ax2.plot(lam_arr*1e9, I/I.max(), color=col, lw=2, label=f'T = {T} K')
    lam_max = b_derived / T
    ax2.axvline(lam_max*1e9, color=col, ls='--', lw=1.2, alpha=0.8)

ax2.axvspan(380, 700, alpha=0.08, color='violet', label='Visible')
ax2.set_xlabel("Wavelength (nm)", fontsize=12)
ax2.set_ylabel("Normalised I(λ, T)", fontsize=12)
ax2.set_title("Planck Curves with Derived λ_max (dashed)
All peaks satisfy λ_max T = b", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle("Problem 6.125 — Deriving Wien's Law from Planck's Law", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Summary

Setting $dI/d\lambda = 0$ in Planck's law and substituting $x = hc/(\lambda k_B T)$ yields the transcendental equation:

$$5(1 - e^{-x}) = x$$

Solving numerically: $x_0 = 4.9651$. Therefore:

$$\boxed{\lambda_{max} T = \frac{hc}{k_B x_0} = \frac{(6.626\times10^{-34})(2.998\times10^8)}{(1.380\times10^{-23})(4.9651)} = 2.898\times10^{-3}\text{ m·K}}$$

This is **Wien's displacement law** — derived from first principles from Planck's radiation law.
